# M5-T7: Deep-Learning Stretch — Fine-tune DistilBERT on Email Text

**Owner:** Sanjeewa Narayana  
**Depends on:** M4-T7 email splits, M5-T2 (classical email winner to beat)

**Run on GPU (Google Colab).** Fine-tune **`distilbert-base-uncased`** on the raw email text and compare it against the best classical email model from M5-T2. DistilBERT is chosen over a CNN because transformers outperform CNNs on natural-language text.

**Bar to beat (M5-T2, classical):** LinearSVC — **F1 = 0.9903**, ROC-AUC = 0.9991. (A very high bar — the interesting question is whether a transformer is *worth the compute* given the classical model is already near-perfect.)

> Colab: `Runtime → Change runtime type → GPU` before running.

## Step 0 — Install + get data
Transformers/datasets on top of Colab's preinstalled torch, and the same M4-T7 email splits M5-T2 used.

In [ ]:
# Install deps first (local conda example):
# conda create -n nlp312 python=3.12 -y
# conda activate nlp312
# conda install -y -c conda-forge pandas scikit-learn scipy jupyterlab ipykernel
# pip install "transformers>=4.40" "datasets>=2.19" accelerate torch
# python -m ipykernel install --user --name nlp312 --display-name "Python (nlp312)"

# Colab install (if needed):
# !pip -q install 'transformers>=4.40' 'datasets>=2.19' accelerate
# !aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/email_train.csv email_train.csv
# !aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/email_test.csv  email_test.csv

import pandas as pd
from pathlib import Path

_base = Path('data/processed') if Path('data/processed/email_train.csv').exists() else Path('.')
tr = pd.read_csv(_base / 'email_train.csv')
te = pd.read_csv(_base / 'email_test.csv')
for d in (tr, te):
    d['text_clean'] = d['text_clean'].fillna('')
print('Train:', tr.shape, '| Test:', te.shape)
print('Label balance (train):', tr['label'].value_counts().to_dict())

Train: (65662, 7) | Test: (16416, 7)
Label balance (train): {1: 34276, 0: 31386}


## Step 1 — Tokenize
`distilbert-base-uncased`, truncate/pad to 256 tokens (emails are short after M4 cleaning). Same train/test rows as every other M5 model.

In [6]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL = 'distilbert-base-uncased'
MAXLEN = 256
tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(df):
    ds = Dataset.from_pandas(
        df[['text_clean', 'label']].rename(columns={'text_clean': 'text'})
    )
    ds = ds.map(
        lambda b: tok(
            b['text'],
            truncation=True,
            padding='max_length',
            max_length=MAXLEN,
        ),
        batched=True,
    )
    drop_cols = [c for c in ds.column_names if c not in ['input_ids', 'attention_mask', 'label']]
    return ds.remove_columns(drop_cols)

ds_train = to_ds(tr)
ds_test  = to_ds(te)
print(ds_train)

Map:   0%|          | 0/65662 [00:00<?, ? examples/s]

Map:   0%|          | 0/16416 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 65662
})


## Step 2 — Fine-tune (2–3 epochs)
HuggingFace `Trainer`. 2 epochs is usually enough for a binary text task this size; bump to 3 if the validation F1 is still climbing.

In [7]:
import time, numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
data_collator = DataCollatorWithPadding(tokenizer=tok)

def hf_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'f1': f1_score(labels, preds)}

args = TrainingArguments(
    output_dir='distilbert-email',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=200,
    seed=42,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    compute_metrics=hf_metrics,
    data_collator=data_collator,
)

t0 = time.time()
trainer.train()
train_time = time.time() - t0
print(f'\nFine-tuned in {train_time:.1f}s')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1
1,0.028753,0.027164,0.993173
2,0.009690,0.029419,0.994164


/opt/anaconda3/envs/venv-lab/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



Fine-tuned in 10829.6s


## Step 3 — Evaluate (M5-T1 metric set) + log
Full metric set on the held-out TEST split, appended to `results/m5_results.csv` next to the classical email models. `cv_f1` is blank (transformers don't use the classical 5-fold CV).

In [8]:
import csv, scipy.special
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

out = trainer.predict(ds_test)
logits = out.predictions
proba = scipy.special.softmax(logits, axis=1)[:, 1]
pred = np.argmax(logits, axis=1)
y_test = te['label'].values

row = {
    'track': 'email',
    'model': 'DistilBERT (fine-tuned)',
    'accuracy':  round(accuracy_score(y_test, pred), 4),
    'precision': round(precision_score(y_test, pred, zero_division=0), 4),
    'recall':    round(recall_score(y_test, pred, zero_division=0), 4),
    'f1':        round(f1_score(y_test, pred, zero_division=0), 4),
    'roc_auc':   round(roc_auc_score(y_test, proba), 4),
    'cv_f1':     '',
    'train_time': round(train_time, 2),
}
print(row)

RESULTS = Path('results/m5_results.csv') if Path('results').exists() else Path('m5_results.csv')
RESULTS.parent.mkdir(parents=True, exist_ok=True)
cols = ['track','model','accuracy','precision','recall','f1','roc_auc','cv_f1','train_time']
exists = RESULTS.exists()
with open(RESULTS, 'a', newline='') as f:
    w = csv.DictWriter(f, fieldnames=cols)
    if not exists: w.writeheader()
    w.writerow(row)

bar = 0.9903  # M5-T2 LinearSVC F1
verdict = 'BEATS' if row['f1'] > bar else 'does NOT beat'
print(f"\nDistilBERT F1 = {row['f1']}  vs  LinearSVC (M5-T2) F1 = {bar}  ->  {verdict} the classical model.")

/opt/anaconda3/envs/venv-lab/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'track': 'email', 'model': 'DistilBERT (fine-tuned)', 'accuracy': 0.9939, 'precision': 0.9944, 'recall': 0.9939, 'f1': 0.9942, 'roc_auc': 0.9997, 'cv_f1': '', 'train_time': 10829.61}

DistilBERT F1 = 0.9942  vs  LinearSVC (M5-T2) F1 = 0.9903  ->  BEATS the classical model.


## Conclusion

- **DistilBERT test F1 = 0.9942** (ROC-AUC 0.9997, precision 0.9944, recall 0.9939) vs classical LinearSVC **0.9903**.
- **Verdict: DistilBERT wins, but only barely — +0.0039 F1 (0.9942 vs 0.9903).** The classical model was already near-perfect, so there is very little headroom left to win.
- **Cost:** ~10,830 s (~3 hours) GPU fine-tuning vs ~42 s CPU for LinearSVC — roughly **258× the training time** for a ~0.4% F1 gain, plus much heavier (GPU/transformer) inference in production.
- **Recommendation: keep LinearSVC for the email track in M6/M7.** The DistilBERT gain is real but far too small to justify the compute and deployment complexity; the classical model is the pragmatic choice. (See the cost-vs-gain plot — DistilBERT sits far to the right at nearly the same height as LinearSVC.)
- **Logged:** row `DistilBERT (fine-tuned)` appended to `results/m5_results.csv` (track=`email`).